# GRIFFIN H2O AutoML Training (A5)
## Multi-class classification: predict DHRM Occupational Family from position description features

**Input**: `workday_features.csv` (103 rows, 15 features from A4)  
**Target**: `occ_family` -- 7 DHRM occupational families  
**Method**: H2O AutoML with 5-fold stratified cross-validation  
**Metrics**: Macro F1, top-3 accuracy, confusion matrix  
**Interpretability**: Variable importance, SHAP (global + local), PDP

---

### Why cross-validation only (no held-out test set)?

With only 103 rows across 7 classes -- some as small as 2-4 observations --
a train/test split would produce unreliable estimates on the minority classes.
H2O's built-in stratified 5-fold CV uses all data for both training and
evaluation, giving more stable performance estimates.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# H2O requires Java. When running inside a conda env, Java may not be on
# the system PATH even though openjdk is installed. Setting JAVA_HOME and
# updating PATH before importing h2o ensures it finds java.exe.
CONDA_PREFIX = os.path.dirname(sys.executable)  # .conda/envs/griffin
JAVA_HOME = os.path.join(CONDA_PREFIX, "Library")
if os.path.isdir(JAVA_HOME):
    os.environ["JAVA_HOME"] = JAVA_HOME
    os.environ["PATH"] = os.path.join(JAVA_HOME, "bin") + os.pathsep + os.environ["PATH"]

import h2o
from h2o.automl import H2OAutoML

h2o.init()

## 1. Load Feature Matrix

In [ ]:
# Paths
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.join(NOTEBOOK_DIR, "..")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

df = pd.read_csv(os.path.join(DATA_DIR, "training", "workday_features.csv"))
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nTarget distribution:")
print(df["occ_family"].value_counts().to_string())

In [ ]:
# Drop 'Unknown' rows (3 rows with NaN job_family -- no true label)
df = df[df["occ_family"] != "Unknown"].copy()
print(f"After dropping Unknown: {df.shape[0]} rows")
print(f"\nClass distribution for modeling:")
print(df["occ_family"].value_counts().to_string())

## 2. Convert to H2OFrame and Set Types

In [ ]:
# Define feature columns and target
feature_cols = [
    "budget_mentioned", "supervises_staff",
    "receives_supervision", "supervision_count",
    "education_level", "years_experience", "text_length",
    "leadership_keywords", "technical_keywords", "research_keywords",
    "healthcare_keywords", "facilities_keywords", "finance_keywords",
    "is_exempt", "is_salaried"
]
target = "occ_family"

# Convert to H2OFrame
hf = h2o.H2OFrame(df[feature_cols + [target]])

# Target must be a factor (categorical) for multi-class classification
hf[target] = hf[target].asfactor()

print(f"H2OFrame: {hf.shape}")
print(f"Target levels: {hf[target].levels()[0]}")
print(f"\nFeature types:")
print(hf.types)

## 3. H2O AutoML Training

AutoML trains and tunes multiple model families (GBM, XGBoost, GLM, DRF,
Deep Learning, stacked ensembles) and ranks them on a leaderboard.

Configuration:
- `max_runtime_secs=300` -- 5 minutes; plenty for 100 rows
- `nfolds=5` -- stratified cross-validation
- `sort_metric="logloss"` -- standard for multi-class; lower is better
- `seed=42` -- reproducibility

In [ ]:
aml = H2OAutoML(
    max_runtime_secs=300,
    nfolds=5,
    seed=42,
    sort_metric="logloss",
    project_name="griffin_occ_family"
)

aml.train(x=feature_cols, y=target, training_frame=hf)
print("AutoML done.")

## 4. Leaderboard

In [ ]:
lb = aml.leaderboard
print(f"Models trained: {lb.nrows}")
lb.head(rows=20)

In [ ]:
best = aml.leader
print(f"Best model: {best.model_id}")
print(f"\nCross-validation metrics:")
perf = best.model_performance(xval=True)
print(perf)

## 5. Confusion Matrix and Per-Class Metrics

In [ ]:
# Cross-validation confusion matrix
cm = perf.confusion_matrix()
print("Confusion Matrix (5-fold CV):")
print(cm)

In [ ]:
# Macro F1 -- average F1 across all classes (treats each class equally)
# H2O reports per-class metrics in the classification table
class_table = perf._metric_json["multinomial_auc_table"]

# Manual macro F1 from confusion matrix
cm_df = cm.to_list()
cm_header = cm.col_header
print("Per-class error rates from CM:")
print(cm)

# Also report overall accuracy
print(f"\nMean per-class error: {perf.mean_per_class_error()}")
print(f"Overall accuracy (1 - mean_per_class_error): {1 - perf.mean_per_class_error():.4f}")

In [ ]:
# Top-3 accuracy: fraction of rows where the true class is among the
# 3 highest predicted probabilities.
#
# Some model types (e.g. selection models) don't store CV holdout predictions.
# In that case, fall back to training-frame predictions with a note.
cv_preds = best.cross_validation_holdout_predictions()

if cv_preds is not None:
    pred_source = "CV holdout"
    pred_df = cv_preds.as_data_frame()
else:
    pred_source = "training frame (CV holdout unavailable for this model type)"
    preds = best.predict(hf)
    pred_df = preds.as_data_frame()

true_labels = df[target].values
prob_cols = [c for c in pred_df.columns if c != "predict"]
prob_matrix = pred_df[prob_cols].values

top3_correct = 0
for i in range(len(true_labels)):
    top3_classes = np.array(prob_cols)[np.argsort(prob_matrix[i])[-3:]]
    if true_labels[i] in top3_classes:
        top3_correct += 1

top3_acc = top3_correct / len(true_labels)
print(f"Top-3 accuracy ({pred_source}): {top3_acc:.4f} ({top3_correct}/{len(true_labels)})")
print(f"  (True class was among the 3 highest-probability predictions)")

## 6. Variable Importance

In [ ]:
# Variable importance from the best model
# Note: stacked ensembles don't have native varimp;
# if the leader is an ensemble, use the best base model instead
if "StackedEnsemble" in best.model_id:
    print("Leader is a Stacked Ensemble -- using best GBM for variable importance.")
    # Find best non-ensemble model
    lb_df = lb.as_data_frame()
    base_models = lb_df[~lb_df["model_id"].str.contains("StackedEnsemble")]
    best_base_id = base_models.iloc[0]["model_id"]
    base_model = h2o.get_model(best_base_id)
    print(f"Best base model: {best_base_id}")
    base_model.varimp_plot(num_of_features=15)
else:
    best.varimp_plot(num_of_features=15)

## 7. SHAP Values

SHAP (SHapley Additive exPlanations) assigns each feature a contribution
score for every prediction. This reveals not just *which* features matter,
but *how* they push predictions toward specific classes.

In [ ]:
# Global SHAP summary
# Use the best non-ensemble model (ensembles don't support SHAP in H2O)
if "StackedEnsemble" in best.model_id:
    shap_model = base_model
else:
    shap_model = best

print(f"SHAP model: {shap_model.model_id}")
shap_model.shap_summary_plot(hf)

In [ ]:
# Local SHAP: explain a single prediction
# Pick an example from the most common class and a minority class
import matplotlib.pyplot as plt

# Example 1: first Administrative Services row
admin_idx = df[df[target] == "Administrative Services"].index[0]
admin_row_num = list(df.index).index(admin_idx)

print(f"Local SHAP for row {admin_row_num} (true: Administrative Services)")
shap_model.shap_explain_row_plot(hf, row_index=admin_row_num)

In [ ]:
# Example 2: first Engineering and Technology row
eng_idx = df[df[target] == "Engineering and Technology"].index[0]
eng_row_num = list(df.index).index(eng_idx)

print(f"Local SHAP for row {eng_row_num} (true: Engineering and Technology)")
shap_model.shap_explain_row_plot(hf, row_index=eng_row_num)

## 8. Partial Dependence Plots

PDPs show the marginal effect of a single feature on the predicted outcome,
averaging over all other features. We plot the top features identified by
variable importance.

In [ ]:
# PDP for top 3 features by variable importance
varimp_df = pd.DataFrame(
    shap_model.varimp(),
    columns=["variable", "relative_importance", "scaled_importance", "percentage"]
)
top3_features = varimp_df["variable"].head(3).tolist()
print(f"Top 3 features: {top3_features}")

for feat in top3_features:
    print(f"\nPDP: {feat}")
    shap_model.pd_plot(hf, column=feat)

## 9. Summary

### Results
- Trained H2O AutoML on 100 position descriptions with 15 engineered features
- Target: 7 DHRM occupational families (multi-class classification)
- Evaluation: 5-fold stratified cross-validation (appropriate for small sample)

### Key Outputs
- **Leaderboard**: ranked models by logloss
- **Confusion matrix**: per-class prediction accuracy
- **Top-3 accuracy**: practical metric for classification suggestions
- **Variable importance**: which features drive predictions
- **SHAP (global)**: how features push predictions across all observations
- **SHAP (local)**: individual prediction explanations
- **PDP**: marginal feature effects

### Next Steps
- Use model predictions as one input to the Streamlit classification app (A6)
- Integrate SHAP explanations into the agentic AI workflow
- Additional visualizations for the Big Data presentation (April 17)

In [ ]:
# Save the best model for later use
model_path = h2o.save_model(model=best, path=os.path.join(PROJECT_ROOT, "models"), force=True)
print(f"Model saved to: {model_path}")

In [ ]:
# Shutdown H2O cluster
# h2o.cluster().shutdown()